# Ensemblr: 2D Grid Umbrella Sampling Setup

This notebook walks through building 2D umbrella sampling windows from an AF2
structural ensemble: load structures, run PCA, optimise a 2D grid of
representative structures over (PC1, PC2), and write PLUMED input files for a
two-CV restrained simulation. All user-configurable parameters are
consolidated in **Section 1 (Setup)**.

For the 1D path-based workflow (single CV, ordered chain of windows), see
`example_notebook_clean.ipynb`. This notebook targets a rectangular grid over
two CVs instead of a single ordered path — use it when you want to restrain
two independent degrees of freedom simultaneously (e.g. two PCs, or two gate
distances) rather than moving along one reaction coordinate.

| Section | Description |
|---------|-------------|
| 1. Setup | Imports and all configurable parameters |
| 2. Load Ensemble | Gather AF2 outputs (or a pre-built structure pool) |
| 3. Selection & PCA | Generate RMSD/PCA selection; run dimensionality reduction |
| 4. Clustering | Identify representative structures via HDBSCAN |
| 5. Structural Properties | RMSD, SASA, and user-defined distances |
| 6. Visualisation | Ensemble density and property plots |
| 7. Property Matrices | Pairwise Ramachandran and CV matrices |
| 8. Grid Construction | Bin structures onto a regular (CV1, CV2) grid |
| 9. Grid Optimisation | Monte Carlo simulated annealing over grid neighbours |
| 10. Grid Assessment | Evaluate window placement quality |
| 11. Umbrella Sampling Setup | Embed structures and write 2D PLUMED input files |
| 12. Debugging | Verify PLUMED CV values against MDA projections |


## 1. Setup

All imports and user-configurable parameters live here. Edit the **SETTINGS**
block before running the notebook — everything below the settings block runs
automatically.

In [ ]:
# ── Suppress warnings / logging BEFORE any imports ───────────────────────────
import warnings
warnings.filterwarnings('ignore')

import os
os.environ["JAX_PLATFORMS"] = "cpu"       # prevent JAX GPU-search spam
os.environ["OMP_NUM_THREADS"] = "1"       # placeholder; overwritten below

# ── Imports ───────────────────────────────────────────────────────────────────
import glob
import json
import pickle
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import jax

from IPython.display import display
from ensemblr import Ensemble, GridPathFinder, GridUmbrellaSamplingPrep
from ensemblr.plumed import verify_2d_pca_cvs

# Silence noisy loggers that bypass Python's warnings system
for _log in ('absl', 'jax', 'jax._src.xla_bridge', 'MDAnalysis', 'numexpr'):
    logging.getLogger(_log).setLevel(logging.ERROR)

# ── SETTINGS ──────────────────────────────────────────────────────────────────

# How to gather the ensemble:
#   'af2_msa_depths' — gather relaxed/unrelaxed PDBs from output_{msa}/ subdirectories
#                       of base_directory (classic generate_af2_ensemble.py layout)
#   'flat'            — PDBs already sit directly in structure_directory, with a
#                       plddt_dict.json alongside (generate_af2_ensemble_v2.py's
#                       adaptive grid-targeted sampler writes exactly this layout —
#                       it is the natural pairing for this notebook, since that
#                       generator already densifies the same PC1/PC2 grid this
#                       notebook optimises windows over)
ensemble_source = 'flat'

# Base directories
base_directory      = "/path/to/ensemble_root/"
run_name             = "run_01"
out_dir              = base_directory + run_name + "/"
structure_directory  = base_directory + "structures/"   # PDBs live here for either source

# Only used when ensemble_source == 'af2_msa_depths'
msa_depths = ['8-16', '16-32', '32-64', '64-128', '128-256', '256-512', '512-1024']
load_relaxed = True   # set False to load _unrelaxed_ PDBs instead of _relaxed_

# Only used when ensemble_source == 'flat'
plddt_dict_path = base_directory + "plddt_dict.json"

# pLDDT threshold — structures below this value are discarded
thresh_pLDDT = 80

# Residue selections for RMSD and PCA calculations
conserved_residues = ""
ignored_residues   = ""
always_include     = ""
include_loops      = True  # set False to exclude loop residues from the PCA selection

# Optional: named inter-atomic distances to compute and add to the ensemble dataframe
distances_to_evaluate = {}   # e.g. {'gate': (sel1, sel2)}

# Optional: external reference structures to compute RMSD against for every ensemble structure.
# Format: {label: absolute_path} — adds a 'rmsd_to_{label}' column to ens.df for each entry.
user_specified_structures = {}   # e.g. {'IF': '/path/to/IF.pdb', 'OF': '/path/to/OF.pdb'}
user_ref_resid_offset     = 0    # residue ID offset applied to user-specified reference structures

# Toggle optional per-structure calculations
calculate_rmsd      = True
calculate_sasa      = False
calculate_distances = False

# CPU cores for parallelised steps
num_processes = 6

# Number of principal components to retain
n_pcs = 3

# Collective variables defining the 2D grid
collective_variable   = 'PC1'   # grid axis 1
collective_variable_2 = 'PC2'   # grid axis 2

# Grid construction — resolution along each axis
grid_n_bins_pc1 = 16
grid_n_bins_pc2 = 16

# Grid Monte Carlo optimisation settings — minimises Ramachandran RMSE (and
# optionally CV mismatch) between 4-connected grid neighbours
grid_n_runs         = 50
grid_n_steps        = 1000
grid_wf_rama        = 1.0   # weight for the Ramachandran continuity term
grid_wf_cv          = 0.0   # weight for the CV-difference term (0 = disabled)
reproducible_seeds  = False

# Template PDB for umbrella sampling (protein + membrane + solvent)
template_pdb_for_umbrella_sampling = '/path/to/template.pdb'
resid_offset    = 0      # offset applied to residue IDs when writing output structures
force_constant  = 1000   # harmonic restraint force constant (kJ mol⁻¹ nm⁻²), both CVs

# Default axes for ensemble plots
plot_variable_1 = 'PC1'
plot_variable_2 = 'PC2'

# ── END SETTINGS ──────────────────────────────────────────────────────────────

os.makedirs(out_dir, exist_ok=True)
os.environ["OMP_NUM_THREADS"] = str(num_processes)

pd.set_option('display.max_colwidth', None)

cmap   = plt.colormaps['tab20c']
colors = cmap(np.linspace(0, 1, 10))
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=colors)

# ── Shared formatting helper ──────────────────────────────────────────────────
def _banner(title, width=64):
    bar = '─' * width
    print(f'\n{bar}\n  {title}\n{bar}')

# ── Summary ───────────────────────────────────────────────────────────────────
_banner("1. Setup")
print(f"  Ensemble source  : {ensemble_source}")
print(f"  Output dir       : {out_dir}")
print(f"  Structures dir   : {structure_directory}")
print(f"  pLDDT threshold  : {thresh_pLDDT}")
print(f"  CPU cores        : {num_processes}")
print(f"  Grid             : {grid_n_bins_pc1} x {grid_n_bins_pc2}  ({collective_variable}, {collective_variable_2})")
print(f"  JAX devices      : {jax.devices()}")

## 2. Load Ensemble

Gathers structures and their pLDDT scores, filters out anything below
`thresh_pLDDT`, and drops structures with fewer than the maximum atom count
to ensure topological consistency. Supports either the classic per-MSA-depth
AF2 output layout, or a flat directory with a `plddt_dict.json` (the layout
written by `generate_af2_ensemble_v2.py`'s adaptive grid-targeted sampler).

In [ ]:
_banner("2. Load Ensemble")

ens = Ensemble(structure_directory, thresh_pLDDT=thresh_pLDDT)

if ensemble_source == 'af2_msa_depths':
    ens.load_from_af2(base_directory, msa_depths, relaxed=load_relaxed)
elif ensemble_source == 'flat':
    with open(plddt_dict_path) as f:
        plddt_dict = json.load(f)
    ens.load(plddt_dict=plddt_dict)
else:
    raise ValueError(f"Unknown ensemble_source: {ensemble_source!r}")

print(f"\n  {len(ens.df)} structures loaded")
print(f"  pLDDT range : {ens.df['pLDDT'].min():.1f} \u2013 {ens.df['pLDDT'].max():.1f}")
#display(ens.df)

## 3. Selection & PCA

Generates an MDAnalysis selection token using DSSP secondary structure
information from the highest-pLDDT structure, then runs PCA on the aligned
ensemble. PC projections are stored in `ens.df`. Visualisation PDBs for each
PC are written to `output_dir`.

In [ ]:
import types

_banner("3. Selection & PCA")

ens.build_selection(
    conserved_residues=conserved_residues,
    excluded_residues=ignored_residues,
    always_include=always_include,
    include_loops=include_loops,
    output_file=out_dir + 'rmsd_selection.pdb',
)
print(f"  Selection : {ens.rmsd_selection}")

_pca_checkpoint = out_dir + 'pca_checkpoint.npz'
pc_cols = [f'PC{i+1}' for i in range(n_pcs)]
_pca_valid = False

if os.path.exists(_pca_checkpoint):
    _npz = np.load(_pca_checkpoint)
    if 'PC1' in _npz and _npz['PC1'].shape[0] == len(ens.df):
        for c in pc_cols:
            ens.df[c] = _npz[c]
        ens.pca = types.SimpleNamespace(
            p_components=_npz['p_components'],
            mean=_npz['mean'],
            cumulated_variance=_npz['cumulated_variance'],
            variance=(_npz['variance'] if 'variance' in _npz
                     else np.diff(np.concatenate([[0], _npz['cumulated_variance']]))),
        )
        ens._ensemble_pdb = out_dir + 'ensemble.pdb'
        cumvar = {i+1: round(float(v)*100, 1) for i, v in enumerate(_npz['cumulated_variance'][:n_pcs])}
        print(f"  Loaded from checkpoint\n  Cumulative variance (%): {cumvar}")
        _pca_valid = True
    else:
        print("  Checkpoint size mismatch \u2014 will recompute")

if not _pca_valid:
    ens.run_pca(n_components=n_pcs, output_dir=out_dir)
    np.savez(
        _pca_checkpoint,
        p_components=ens.pca.p_components,
        mean=np.asarray(ens.pca.mean).flatten(),
        cumulated_variance=np.asarray(list(ens.pca.cumulated_variance)),
        variance=np.asarray(list(ens.pca.variance[:n_pcs])),
        **{c: ens.df[c].values for c in pc_cols},
    )
    print(f"  Saved checkpoint: {_pca_checkpoint}")

print(f"\n  PC projections (first 5 rows):")
#display(ens.df[pc_cols].head())

## 4. Clustering

Clusters the ensemble in PC space using HDBSCAN (minimum samples = 10% of
ensemble size). Cluster medoids are used as representative structures and
become the default RMSD reference set.

In [ ]:
_banner("4. Clustering")

ref_structures = ens.cluster()

num_clusters = ens.df['cluster'].nunique() - (1 if -1 in ens.df['cluster'].values else 0)
n_noise      = (ens.df['cluster'] == -1).sum()

print(f"\n  Clusters : {num_clusters}  |  Noise points : {n_noise}")
print(f"  Medoids  : {ref_structures}")

if 'cluster' in ens.df.columns:
    g = sns.PairGrid(ens.df[pc_cols + ['cluster']], vars=pc_cols, hue='cluster',
                     height=1.5, palette='Paired')
    g.map(plt.scatter, marker='.', s=2)
    for rep in ref_structures:
        for i in range(len(pc_cols)):
            for j in range(len(pc_cols)):
                g.axes[i][j].scatter(
                    ens.df.loc[rep, pc_cols[j]], ens.df.loc[rep, pc_cols[i]],
                    c='black', marker='x', s=100, zorder=10,
                )
else:
    g = sns.PairGrid(ens.df[pc_cols], height=1.5)
    g.map(plt.scatter, marker='.', s=2)
plt.suptitle('PC space', y=1.02)
plt.show()

## 5. Structural Properties

Computes per-structure properties in parallel and adds them as columns to
`ens.df`: RMSD to each reference structure, solvent-accessible surface area,
and user-defined inter-atomic distances.

In [ ]:
_banner("5. Structural Properties")

_df_checkpoint = out_dir + 'ensemble_df.parquet'

_want_rmsd  = [f'rmsd_to_{r}' for r in ref_structures] if calculate_rmsd else []
_want_dists = list(distances_to_evaluate.keys())        if calculate_distances else []
_want_sasa  = ['sasa']                                  if calculate_sasa else []
_want_refs  = [f'rmsd_to_{k}' for k in user_specified_structures] if user_specified_structures else []
_want_all   = _want_rmsd + _want_dists + _want_sasa + _want_refs

# Restore any already-computed columns that are still valid for this ensemble
if os.path.exists(_df_checkpoint):
    _saved = pd.read_parquet(_df_checkpoint)
    if _saved.index.equals(ens.df.index):
        for c in [c for c in _want_all if c in _saved.columns and c not in ens.df.columns]:
            ens.df[c] = _saved[c]
    else:
        print(f"  Note: checkpoint has {len(_saved)} structures vs {len(ens.df)} \u2014 will recompute")

# Compute anything still missing (covers first run, changed threshold, changed ref structures)
_missing_rmsd  = [r for r in ref_structures if f'rmsd_to_{r}' not in ens.df.columns]
_missing_dists = {k: v for k, v in distances_to_evaluate.items()
                  if k not in ens.df.columns and all(v)} if calculate_distances else {}
_missing_sasa  = calculate_sasa and 'sasa' not in ens.df.columns
_missing_refs  = {k: v for k, v in user_specified_structures.items()
                  if f'rmsd_to_{k}' not in ens.df.columns} if user_specified_structures else {}

if _missing_rmsd or _missing_dists or _missing_sasa:
    ens.compute_metrics(
        ref_structures=_missing_rmsd,
        distances=_missing_dists or None,
        compute_sasa=_missing_sasa,
        num_processes=num_processes,
    )
    ens.df.to_parquet(_df_checkpoint)
    print(f"  Saved \u2192 {_df_checkpoint}")

if _missing_refs:
    ens.compute_reference_rmsds(_missing_refs, ref_resid_offset=user_ref_resid_offset,
                                num_processes=num_processes)
    ens.df.to_parquet(_df_checkpoint)
    print(f"  Saved \u2192 {_df_checkpoint}")

if not (_missing_rmsd or _missing_dists or _missing_sasa or _missing_refs):
    print(f"  Loaded from checkpoint")

print(f"\n  Ensemble dataframe  ({len(ens.df)} rows \u00d7 {len(ens.df.columns)} cols):")
#display(ens.df)

# ── Closest structures to user-specified references ──────────────────────────
if user_specified_structures:
    print("\n  Closest ensemble structures to user-specified references:")
    for label in user_specified_structures:
        col = f'rmsd_to_{label}'
        closest = ens.df[col].idxmin()
        print(f"    {label}: {closest}  (RMSD = {ens.df.loc[closest, col]:.2f} \u00c5)")

## 6. Visualisation

Plots ensemble density in PC space (hexbin), overlaid distance distributions,
RMSD histograms, and SASA/pLDDT distributions. Reference structures are
annotated with crosses.

In [ ]:
_banner("6. Visualisation")

valid_distances = []
if calculate_distances and distances_to_evaluate:
    valid_distances = [d for d in distances_to_evaluate if not ens.df[d].isna().all()]

n_hex_panels = 1 + len(valid_distances)
n_hist       = (1 if calculate_rmsd else 0) + (1 if calculate_sasa else 0) + 1
ncols        = n_hex_panels + n_hist

fig, axes = plt.subplots(1, ncols, figsize=(3 * ncols, 3))
axes = np.atleast_1d(axes)

# ── hexbin panels ──────────────────────────────────────────────────────────
hb = axes[0].hexbin(ens.df[plot_variable_1], ens.df[plot_variable_2],
                    cmap='plasma', gridsize=50, mincnt=1, bins='log')
axes[0].set_xlabel(plot_variable_1); axes[0].set_ylabel(plot_variable_2)
fig.colorbar(hb, ax=axes[0]); axes[0].set_title('Density')
for ref in ref_structures:
    axes[0].scatter(ens.df.loc[ref, plot_variable_1], ens.df.loc[ref, plot_variable_2],
                    c='black', marker='x', s=100)

for i, distance_name in enumerate(valid_distances):
    hb = axes[1 + i].hexbin(ens.df[plot_variable_1], ens.df[plot_variable_2],
                             C=ens.df[distance_name], cmap='coolwarm', gridsize=50, mincnt=1)
    axes[1 + i].set_xlabel(plot_variable_1); axes[1 + i].set_ylabel(plot_variable_2)
    fig.colorbar(hb, ax=axes[1 + i], label=distance_name)
    axes[1 + i].set_title(distance_name)

# ── histogram panels ────────────────────────────────────────────────────────
ax_idx = n_hex_panels

if calculate_rmsd:
    for ref in ref_structures:
        axes[ax_idx].hist(ens.df[f'rmsd_to_{ref}'], bins=20, alpha=0.5, label=ref)
    axes[ax_idx].set_xlabel('RMSD'); axes[ax_idx].set_ylabel('Count')
    axes[ax_idx].set_xlim(0)
    ax_idx += 1

if calculate_sasa:
    axes[ax_idx].hist(ens.df['sasa'], bins=20, alpha=0.5, color='C3')
    axes[ax_idx].set_xlabel('SASA'); axes[ax_idx].set_ylabel('Count')
    ax_idx += 1

axes[ax_idx].hist(ens.df['pLDDT'], bins=20, alpha=0.5)
axes[ax_idx].set_xlabel('pLDDT')

plt.tight_layout()
plt.show()

## 7. Property Matrices

Builds pairwise matrices over the ensemble required as inputs to the grid
Monte Carlo optimisation: an all-vs-all Ramachandran circular-distance matrix
(step-cost between grid neighbours), and CV difference matrices for
`collective_variable` / `collective_variable_2`. Grid optimisation always
uses the Ramachandran matrix — it is a much cheaper continuity metric than
RMSD to compute pairwise and does not depend on rigid-body alignment.

In [ ]:
_banner("7. Property Matrices")

_matrices_checkpoint = out_dir + 'matrices_rama.npz'
_n = len(ens.df)
_matrices_valid = False

if os.path.exists(_matrices_checkpoint):
    _npz = np.load(_matrices_checkpoint)
    if 'rama_matrix' in _npz and _npz['rama_matrix'].shape[0] == _n:
        ens.rama_matrix = _npz['rama_matrix']
        ens.cv_matrix   = _npz['cv_matrix']
        ens.cv2_matrix  = _npz['cv2_matrix'] if 'cv2_matrix' in _npz else None
        _matrices_valid = True
        print(f"  Loaded from checkpoint")
    else:
        print(f"  Checkpoint size mismatch \u2014 will recompute")

if not _matrices_valid:
    ens.compute_matrices(cv_column=collective_variable, cv2_column=collective_variable_2,
                         step_matrix='rama')
    _save = {'rama_matrix': ens.rama_matrix, 'cv_matrix': ens.cv_matrix}
    if ens.cv2_matrix is not None:
        _save['cv2_matrix'] = ens.cv2_matrix
    np.savez(_matrices_checkpoint, **_save)
    print(f"  Saved \u2192 {_matrices_checkpoint}")

print(f"  Rama matrix : {ens.rama_matrix.shape}  min={ens.rama_matrix.min():.2f}  max={ens.rama_matrix.max():.2f}")

matrices = [ens.cv_matrix, ens.rama_matrix]
titles   = ['CV matrix', 'RAMA matrix']
if ens.cv2_matrix is not None:
    matrices.append(ens.cv2_matrix); titles.append('CV2 matrix')

fig, axes = plt.subplots(1, len(matrices), figsize=(3 * len(matrices), 3))
axes = np.atleast_1d(axes)
for ax, matrix, title in zip(axes, matrices, titles):
    ax.imshow(matrix, cmap='inferno'); ax.set_title(title)
plt.tight_layout()
plt.show()

## 8. Grid Construction

Bins structures onto a regular `grid_n_bins_pc1` × `grid_n_bins_pc2` grid over
(`collective_variable`, `collective_variable_2`), and picks the structure
closest to each cell centre as the initial window assignment.

In [ ]:
_banner("8. Grid Construction")

gfinder = GridPathFinder(
    ens, n_bins_pc1=grid_n_bins_pc1, n_bins_pc2=grid_n_bins_pc2,
    cv1=collective_variable, cv2=collective_variable_2,
)
gfinder.build_grid()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3.5))

hb = ax1.hexbin(ens.df[plot_variable_1], ens.df[plot_variable_2],
                gridsize=40, cmap='plasma', mincnt=1, bins='log')
ax1.set_xlabel(plot_variable_1); ax1.set_ylabel(plot_variable_2)
fig.colorbar(hb, ax=ax1); ax1.set_title('Ensemble density')

occ = np.zeros((grid_n_bins_pc1, grid_n_bins_pc2))
for (i, j), members in gfinder._bin_members.items():
    occ[i, j] = len(members)
im = ax2.imshow(
    occ.T, origin='lower', cmap='viridis', aspect='auto',
    extent=[gfinder._pc1_centers[0], gfinder._pc1_centers[-1],
            gfinder._pc2_centers[0], gfinder._pc2_centers[-1]],
)
fig.colorbar(im, ax=ax2, label='structures / cell')
ax2.set_title('Grid occupancy')
ax2.set_xlabel(collective_variable); ax2.set_ylabel(collective_variable_2)

plt.tight_layout()
plt.show()

## 9. Grid Optimisation

Runs `grid_n_runs` independent Monte Carlo simulated annealing optimisations
in parallel. The energy function penalises large Ramachandran-distance steps
(and optionally CV mismatch) between 4-connected grid neighbours, so
adjacent windows stay structurally continuous. Results are sorted by total
grid energy.

In [ ]:
_banner("9. Grid Optimisation \u2014 Monte Carlo")

_grid_checkpoint = out_dir + 'grid_mc_results.pkl'

if os.path.exists(_grid_checkpoint):
    with open(_grid_checkpoint, 'rb') as f:
        _saved_runs = pickle.load(f)
    _saved_structs = {s for _, grid, _ in _saved_runs for s in grid.values()}
    _ens_structs   = set(ens.df.index)
    if _saved_structs.issubset(_ens_structs):
        gfinder.runs_list = _saved_runs
        gfinder.best_grid = _saved_runs[0][1]
        print(f"  Loaded from checkpoint : {_grid_checkpoint}")
    else:
        n_missing = len(_saved_structs - _ens_structs)
        print(f"  Note: grid checkpoint has {n_missing} unknown structures \u2014 will rerun")

if gfinder.runs_list is None:
    gfinder.run(
        n_runs=grid_n_runs, n_steps=grid_n_steps, num_processes=num_processes,
        reproducible_seeds=reproducible_seeds,
        wf_rama=grid_wf_rama, wf_cv=grid_wf_cv,
    )
    with open(_grid_checkpoint, 'wb') as f:
        pickle.dump(gfinder.runs_list, f)
    print(f"  Saved checkpoint \u2192 {_grid_checkpoint}")

print(f"\n  Best  grid energy : {gfinder.runs_list[0][0]:.4f}")
print(f"  Worst grid energy : {gfinder.runs_list[-1][0]:.4f}")

fig, ax = plt.subplots(figsize=(5, 3))
ax.hist([r[0] for r in gfinder.runs_list], bins=20, color='steelblue', alpha=0.7)
ax.axvline(gfinder.runs_list[0][0], color='black', linestyle='dashed', label='best')
ax.set_xlabel('Grid energy (weighted RMSE)'); ax.set_ylabel('Runs')
ax.set_title('MC run energy distribution'); ax.legend()
plt.tight_layout()
plt.show()

## 10. Grid Assessment

Visualises the best grid assignment in PC space and compares actual CV
values at each window against the grid-cell centre. Large offsets suggest
the MC struggled in that region — consider increasing `grid_n_bins_pc1` /
`grid_n_bins_pc2` occupancy (more structures per cell) or `grid_n_runs`.

In [ ]:
_banner("10. Grid Assessment")

summary = gfinder.get_grid_summary()
print(f"  Windows : {len(summary)}")
#display(summary.head())

actual_pc1 = np.array([ens.df.loc[s, collective_variable]   for s in summary['structure']])
actual_pc2 = np.array([ens.df.loc[s, collective_variable_2] for s in summary['structure']])
offset = np.sqrt((actual_pc1 - summary['pc1_target'].values) ** 2
                 + (actual_pc2 - summary['pc2_target'].values) ** 2)

print(f"  Mean offset from cell centre : {offset.mean():.4f}")
print(f"  Max  offset from cell centre : {offset.max():.4f}  (window {int(offset.argmax())})")

fig, axs = plt.subplots(1, 3, figsize=(12, 3.5))

# panel 1: selected windows over ensemble density
hb = axs[0].hexbin(ens.df[plot_variable_1], ens.df[plot_variable_2],
                   gridsize=40, cmap='plasma', mincnt=1, bins='log')
axs[0].scatter(summary['pc1_target'], summary['pc2_target'], c='black', s=10)
axs[0].set_xlabel(plot_variable_1); axs[0].set_ylabel(plot_variable_2)
axs[0].set_title('Selected windows')

# panel 2: target cell centre vs actual structure CV values
axs[1].scatter(summary['pc1_target'], summary['pc2_target'],
               c='black', marker='x', s=40, label='target (cell centre)')
sc = axs[1].scatter(actual_pc1, actual_pc2, c=offset, cmap='coolwarm', s=30, label='actual (structure)')
fig.colorbar(sc, ax=axs[1], label='offset')
axs[1].set_xlabel(collective_variable); axs[1].set_ylabel(collective_variable_2)
axs[1].legend(fontsize=7); axs[1].set_title('Target vs actual window position')

# panel 3: per-window offset histogram
axs[2].hist(offset, bins=20, color='black', alpha=0.5)
axs[2].axvline(offset.mean(), color='red', linestyle='dashed', label=f'mean={offset.mean():.3f}')
axs[2].set_xlabel('Distance from cell centre'); axs[2].set_ylabel('Windows')
axs[2].set_title('Window placement offset'); axs[2].legend()

plt.tight_layout()
plt.show()

## 11. Umbrella Sampling Setup

Embeds each window structure into the simulation template (protein + lipids
+ water) by replacing the protein coordinates, then writes 2D PLUMED input
files (two PCA restraints per window) for both multi-directory and
single-window runs.

**Note:** Always inspect output windows for clashes, particularly threaded
lipids. The template should come from a prior simulation with a working
topology.

In [ ]:
_banner("11. Umbrella Sampling Setup")

prep = GridUmbrellaSamplingPrep(
    grid_finder=gfinder,
    ensemble=ens,
    template_pdb=template_pdb_for_umbrella_sampling,
    output_dir=out_dir + 'umbrella_sampling_2d/',
)

prep.prepare_windows(resid_offset=resid_offset)
print(f"\n  Windows written \u2192 {out_dir}umbrella_sampling_2d/")

prep.write_plumed_input(force_constant=force_constant, resid_offset=resid_offset)
print(f"  PLUMED input written")

## 12. Debugging (Optional)

Runs `plumed driver` on each window's structure/trajectory and compares the
CV1/CV2 values PLUMED reports against the MDA-computed PC projections. Run
after an initial short simulation (or even on the freshly-embedded window
PDBs) to verify PLUMED tracks the expected CVs before committing to a full
production run.

In [ ]:
_banner("12. Debugging \u2014 PLUMED CV verification")

plumed_exe = 'plumed'    # path to the plumed executable
plumed_lib = None        # e.g. '.../plumed/2.8.1/lib' if needed to avoid libstdc++ conflicts
gro_name   = 'system.gro'  # trajectory/structure filename inside each window dir

window_dirs = sorted(
    glob.glob(out_dir + 'umbrella_sampling_2d/window_*'),
    key=lambda p: int(p.rstrip('/').split('_')[-1]),
)
window_targets = gfinder.get_window_targets() if gfinder.best_grid is not None else []
structures = [t[0] for t in window_targets]

if not window_dirs or not structures:
    print("  No window directories found \u2014 run Section 11 (Umbrella Sampling Setup) "
          "and a short test simulation first.")
else:
    result = verify_2d_pca_cvs(
        window_dirs, structures, collective_variable, collective_variable_2,
        ens.df, plumed_exe, gro_name=gro_name, plumed_lib=plumed_lib,
    )

    valid = result['valid']
    print(f"  Valid windows : {valid.sum()}/{len(valid)}")

    if valid.any():
        dev1 = np.abs(result['plumed_cv1'][valid] - result['mda_cv1'][valid])
        dev2 = np.abs(result['plumed_cv2'][valid] - result['mda_cv2'][valid])
        print(f"  Max |PLUMED \u2212 MDA| deviation  CV1: {dev1.max():.4f}   CV2: {dev2.max():.4f}")

        fig, axs = plt.subplots(1, 2, figsize=(8, 3.5))
        for ax, plumed_vals, mda_vals, label in (
            (axs[0], result['plumed_cv1'][valid], result['mda_cv1'][valid], collective_variable),
            (axs[1], result['plumed_cv2'][valid], result['mda_cv2'][valid], collective_variable_2),
        ):
            ax.scatter(mda_vals, plumed_vals, s=20, alpha=0.7)
            lo, hi = min(mda_vals.min(), plumed_vals.min()), max(mda_vals.max(), plumed_vals.max())
            ax.plot([lo, hi], [lo, hi], color='black', linestyle='dashed', alpha=0.5)
            ax.set_xlabel(f'MDA {label}'); ax.set_ylabel(f'PLUMED {label}')
            ax.set_title(label)
        plt.tight_layout()
        plt.show()